# 01 — Rebuild the Factor Lab research dataset

Run only after Notebook 00 has no session mismatch. The result is a Factor Lab-owned, revisioned daily dataset reconstructed from selected all-contract 1min data.

In [ ]:
from pathlib import Path
import json
from saltlab.factorlab import SaltDataConfig, SaltDataMinuteRepository

lab_root = Path.cwd().resolve().parent
repo = SaltDataMinuteRepository(SaltDataConfig(root=Path.home() / 'Developer' / 'salt-data'))
snapshot = repo.snapshot()
daily, audit = repo.build_daily()
assert len(audit) == 87
assert audit['session_rows_unmatched'].sum() == 0
assert set(daily['mapping_source']) <= {'source_schedule', 'verified_main_outside_schedule'}
daily.shape, audit['status'].value_counts()

In [ ]:
coverage = repo.coverage_by_year(daily)
run_dir = lab_root / 'outputs' / 'datasets' / snapshot.catalog_revision
run_dir.mkdir(parents=True, exist_ok=True)
daily.to_parquet(run_dir / 'daily_reconstructed.parquet', index=False)
audit.to_parquet(run_dir / 'product_build_audit.parquet', index=False)
coverage.to_parquet(run_dir / 'coverage_by_year.parquet', index=False)
(run_dir / 'manifest.json').write_text(json.dumps({
    'catalog_revision': snapshot.catalog_revision,
    'catalog_schema_version': snapshot.schema_version,
    'universe_policy': repo.config.universe_policy,
    'mapping_policy': repo.config.main_mapping_policy,
    'timestamp_semantics': repo.config.timestamp_semantics,
    'rows': len(daily),
}, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
display(coverage)
run_dir